## Setup

### Load Modules

In [3]:
%load_ext autoreload
%autoreload 2

#General Import
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
from os.path import join
from sklearn.model_selection import KFold
from matplotlib import colormaps as cmaps
from mne.filter import filter_data, resample
import scipy.stats as stats
import pandas as pd
from itertools import product
import xarray as xr
from scipy.signal import coherence, welch, hilbert
from scipy.ndimage import convolve1d
from joblib import Parallel, delayed

#ML Import
from sklearn.decomposition import PCA, FastICA, SparsePCA, FactorAnalysis
from sklearn.preprocessing import StandardScaler, scale
import jPCA
import scipy.signal as signal
from statsmodels.tsa.stattools import grangercausalitytests
import tslearn

#Electrophysiology Import
from spyeeg.models.TRF import TRFEstimator
from spyeeg.models.ERP import ERP_class
from spyeeg.utils import lag_matrix
import mne
import frites
from frites.simulations import sim_multi_suj_ephy
from frites.dataset import DatasetEphy
from frites.workflow import WfConnComod
from frites import set_mpl_style
import frites.conn as conn
from scipy.signal import welch
import spectral_connectivity 
from frites.simulations import sim_multi_suj_ephy, sim_mi_cc
from frites.dataset import DatasetEphy
from frites.workflow import WfMi, WfMiCombine
from frites import set_mpl_style

#Graph Import
import networkx as nx


#Performance Import
import time
import psutil

#Local Import
from stats_utils import cliffs_delta, cohen_d, select_clusters
from nice_utils import estimate_loop_time, decorator_loop
from preprocessing_utils import mono_to_bipolar, select_channels, available_regions, delete_channels, adj_scale, ica_shaft, pick_channels
from signal_utils import sparse_resample, lag_finder, sparse_realign
from viz_utils import create_matshow_gif, create_collection_gif, _arrow3D

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load Features

In [5]:
# Acoustic Regressors
fs = 100
new_path = 'C:/Users/D-CAP/Documents/GitHub/witching-star/regressors/selected_regs.pkl'
new_data = pickle.load(open(new_path, 'rb'))
data_fs = new_data['fs']
new_regressors = new_data['regs']
new_names = new_data['regs_name']
ratio = fs/data_fs
new_duration = int(new_regressors.shape[0] * ratio) + 1

new_resamp = []
for i in range(new_regressors.shape[1]):
    name = new_names[i]
    if name in ['Intensity', 'Envelope Oganian', 'Envelope Derivative TF', 'F0 Loudness', 'SpectralFlux Filtered', 'SpectralFlux not_filtered']:
        new_reg = mne.filter.resample(new_regressors[:,i], up=100, down=data_fs)[:new_duration]
    elif name in ['peakEnv_tf', 'Syllabe Onset', 'p-syl', 'Phono']:
        new_reg = new_regressors[:,i] - np.min(new_regressors[:,i])
        new_reg = sparse_resample(new_reg, new_fs = fs, current_fs = data_fs)[:new_duration]
    else:
        print('wut')
    new_resamp.append(new_reg)
new_resamp = np.asarray(new_resamp).T
regressors = new_resamp
regressors_name = new_data['regs_name']


In [6]:
# Renyi2 Array

path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/renyi_array2.pickle"
renyi_data = pickle.load(open(path_renyi, 'rb'))
data_fs = renyi_data['fs']
X = np.roll(renyi_data['X'],4, axis=0)
onsets = np.where(X[:,5] >0)[0]
reg_renyi = np.zeros([regressors.shape[0], X.shape[1]])

for onset in onsets:
    new_onset = int(onset/data_fs*fs)
    for renyi_index in range(X.shape[1]):
        reg_renyi[new_onset, renyi_index] = X[onset,renyi_index]

regressors = np.hstack([regressors, reg_renyi])
regressors_name = regressors_name + renyi_data['names']

### Load Neural Data

In [ ]:
# Load Broadband

data_subject = dict()
channels_subject = dict()
locations_subject = dict()
path_data = "D:/DataSEEG_Sorciere/BIDS/data_mne_fif"
for index_subject in range(1,40):
    try:
        name_subject = 'sub-{:03}'.format(index_subject)
        name_file = name_subject + '_task-iSpeech_speech-epo.fif'
        path_file = os.path.join(path_data, name_subject,'preprocessed','epochs',name_file)
        if os.path.isfile(path_file):
            mne_data = mne.read_epochs(path_file, verbose = False)
        elif os.path.isfile(os.path.join(path_data, name_subject,'preprocessed','monopolar',name_file)):
            path_file = os.path.join(path_data, name_subject,'preprocessed','monopolar',name_file)
            mne_data = mne.read_epochs(path_file, verbose = False)
        else:
            path_file = os.path.join(path_data, name_subject,'preprocessed','epochs','monopolar',name_file)
            mne_data = mne.read_epochs(path_file, verbose = False)
        mne_data_resample = mne_data.resample(fs, npad = 'auto', verbose = False)
        mne_data_resample.filter(0.3, 49, verbose = False)
        channels = mne_data_resample.ch_names
        montage = mne_data_resample.get_montage()
        ch_names = [n for n,_ in montage.get_positions()['ch_pos'].items()] 
        loc = (1e3 * np.stack([coord for _,coord in montage.get_positions()['ch_pos'].items()])).T  # store locations
        
        data_subject[index_subject] = mne_data_resample.get_data(copy = False)[0].T[:regressors.shape[0],:]
        channels_subject[index_subject] = channels
        locations_subject[index_subject] = loc
        print('Subject', index_subject, 'loaded')
    except:
        print('Error in subject', index_subject)

data_bipolar, channels_bipolar, locations_bipolar = mono_to_bipolar(data_subject, channels_subject, locations_subject)

## Compute MI

In [ ]:
#all_channel_selections = [['H','T'], ['H','T','GPH', 'B', 'IF', 'TP', 'STG']]
#all_channel_selections = [['H','T','OP', 'OT', 'OR', 'OC', 'TP', 'TB', 'GPH']]
all_channel_selections = [['H', 'T']]
all_channel_selections = [[]]
n_perm = 1000

exclude = True
montage = 'bipo'  #'bipo' , 'Hfa_bipo'
mi_type = 'cc' #cc(continuous), cd(discrete values) 
inference = 'ffx' #ffx
cluster_th = 'tfce'
#cluster_th = 0.002 #0.002
#kernel = np.hanning(3) #10
kernel = np.ones(3)
kw = dict(n_jobs=1, n_perm=n_perm)
mcp = 'cluster'
tmin = -1.0 #-0.5
tmax = 1.3 #1.3
step = 1
env_ref = 1
baseline_limits = [0,50]
apply_baseline = True
baseline_str = (not apply_baseline) * 'no_baseline'
time_array = np.linspace(tmin, tmax,int((tmax-tmin)*fs))

for channel_selection in all_channel_selections:
    channel_selection_join = ''.join(channel_selection)
    #for regressor_index in [233,253] + list(np.arange(234,253)) + [258]:
    #for regressor_index in [881]:
    #for regressor_index in [736,735,737]:
    for regressor_index in [893,894]:
        for subject_index in range(len(data_subject)):
            ii_dict = dict()
            iimono_dict = dict()
            subject_id = list(data_subject.keys())[subject_index]
            eeg = data_bipolar[subject_id]
            channels = channels_bipolar[subject_id]
            eeg_mono = data_subject[subject_id]
            channels_mono = channels_subject[subject_id]
            eeg_Hfa = data_Hfabipolar[subject_id]
            channels_Hfa = channels_bipolar[subject_id]
            
            eeg_HT, channels_HT = select_channels(eeg,channels, channel_select = channel_selection, exclude = exclude)
            eeg_mono_HT, channels_mono_HT = select_channels(eeg_mono,channels_mono, channel_select = channel_selection, exclude = exclude)
            eeg_Hfa_HT, channels_Hfabipo_HT = select_channels(eeg_Hfa,channels, channel_select = channel_selection, exclude = exclude)
            print('\nSubject:',subject_index, '\nRegressor ID:', regressor_index)
            print("Computing MI over ", len(channels_mono_HT), "channels")
            y1 = eeg_HT[:,:]
            y1_mono = eeg_mono_HT[:,:]
            y1_Hfa = eeg_Hfa_HT[:,:]
            x1 = regressors[:y1.shape[0],[regressor_index]]

            signal = x1[:,0]

            erp = ERP_class(tmin = tmin, tmax = tmax, srate=100)

            if montage == 'mono':
                erp.add_events(y1_mono, signal, weight_events = False, record_weight = True)
                channels_name = channels_mono_HT
            elif montage == 'bipo':
                erp.add_events(y1, signal, weight_events = False, record_weight = True)
                channels_name = []
                for chan_name in channels_HT:
                    channels_name.append(chan_name.replace('-', '||'))
            elif montage == 'ica':                
                y_ica, channels_ica = ica_shaft(y1_mono, channels_mono_HT, random_state = 0)
                erp.add_events(y_ica, signal, weight_events = False, record_weight = True)
                channels_name = channels_ica
            elif montage == 'Hfa_bipo':
                erp.add_events(y1_Hfa, signal, weight_events = False, record_weight = True)
                channels_name = []
                for chan_name in channels_Hfabipo_HT:
                    channels_name.append(chan_name.replace('-', '||'))

            epoched_data = np.asarray(erp.evoked).transpose(0,2,1)
            if apply_baseline:
                baseline = np.repeat(epoched_data[:,:,baseline_limits].mean(-1), epoched_data.shape[-1]).reshape(epoched_data.shape)
                epoched_data = epoched_data - baseline
            epoched_reg = np.asarray(erp.weights)
            if mi_type == 'cd':
                epoched_reg = epoched_reg.astype(int)
            #data = [epoched_data[:,:,baseline_limits[1]:]]
            #time_array_mi = time_array[baseline_limits[1]:]
            data = [epoched_data[:,:,:]]
            time_array_mi = time_array[:]
            y = [epoched_reg]
            roi = [channels_name]
            dt = DatasetEphy(data, y=y, roi=roi, times=time_array_mi, verbose=False)
            wf = WfMi(mi_type, inference, verbose=False, kernel=kernel)
            mi, pvalues = wf.fit(dt, mcp=mcp, cluster_th=cluster_th, **kw)
            mi_p = np.asarray(wf.mi_p)
            mi_stat = {'mutual': mi, 'pval': pvalues, 'roi': channels_name, 'times': time_array_mi, 'permut': mi_p}

            #filename = 'MIStat/MIStat_' + str(n_perm) + '_' + montage + '_' + channel_selection_join + '_reg' + str(regressor_index) + '_sub' + str(subject_id) + '.pickle'
            #filename = 'D:/Data_MI/MIStat_' + str(n_perm) + '_' + montage + '_' + channel_selection_join + '_reg' + str(regressor_index) + '_sub' + str(subject_id) + '.pickle'
            filename = 'D:/Data_MI/MIStat3_' + baseline_str + str(n_perm) + '_' + montage + '_' + channel_selection_join + '_reg' + str(regressor_index) + '_sub' + str(subject_id) + '.pickle'
            
            with open(filename, 'wb') as file:
                pickle.dump(mi_stat, file)
            print("==========")
            